# Сравнение MLP, RNN и CNN для классификации изображений

В данной работе проводится сравнительный анализ различных архитектур нейронных сетей.
Цель исследования — определить влияние архитектуры модели на качество классификации изображений.

## Постановка задачи
Задача заключается в классификации медицинских изображений с использованием различных типов нейронных сетей.

В рамках работы рассматриваются три архитектуры:
- Полносвязная нейронная сеть (MLP)
- Рекуррентная нейронная сеть (RNN)
- Сверточная нейронная сеть (CNN)

Все модели обучаются и тестируются на одном и том же датасете, что позволяет провести корректное сравнение их эффективности.

## Датасет

В работе используется датасет Medical MNIST, содержащий медицинские изображения, разделённые на несколько классов.

Перед подачей в модель изображения проходят этап предобработки:
- изменение размера
- преобразование в тензоры
- нормализация

## Загрузка и предобработка данных

На данном этапе выполняется загрузка датасета и применение преобразований к изображениям.

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

from models.mlp import MLP
from models.cnn import CNN
from models.rnn import RNN

from utils.train import train
from utils.evaluate import evaluate
from utils.dataset import get_dataloaders, get_class_names, get_dataset_info

def run_experiment(model, name, epochs=5, lr=1e-3):

    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses = []

    for epoch in range(epochs):
        loss = train(model, train_loader, optimizer, criterion, device)
        train_losses.append(loss)

        print(f"{name} | Epoch {epoch+1}/{epochs} | Loss: {loss:.4f}")

    acc = evaluate(model, test_loader, device)

    return acc, train_losses

device = torch.device("mps" if torch.backends.mps.is_built() else "cpu")
torch.manual_seed(42)

data_dir = "data/Medical_MNIST"
dataset_info = get_dataset_info(data_dir)

img_size = dataset_info["height"]
input_size = img_size * img_size
gray_scale = (dataset_info["channels"] == 1)

train_loader, test_loader = get_dataloaders(data_dir, batch_size=32, image_size=img_size, gray_scale=gray_scale)
classes = get_class_names(data_dir)

print(classes)

{'channels': 1, 'height': 64, 'width': 64}
torch.Size([32, 1, 64, 64])
['AbdomenCT', 'BreastMRI', 'CXR', 'ChestCT', 'Hand', 'HeadCT']


## Эксперимент 1: Полносвязная нейронная сеть (MLP)

Полносвязная нейронная сеть рассматривает изображение как одномерный вектор признаков.

Недостатком данного подхода является игнорирование пространственной структуры изображения.

In [ ]:
mlp = MLP(input_size=input_size)
mlp_acc, mlp_loss = run_experiment(mlp, "MLP")

: 

## Эксперимент 2: Рекуррентная нейронная сеть (RNN)

Для применения рекуррентной сети изображение преобразуется в последовательность (например, по строкам).

Данный подход позволяет использовать временные зависимости, однако не учитывает двумерную структуру изображения в полной мере.

In [ ]:
rnn = RNN(input_size=img_size, hidden_size=img_size)
rnn_acc, rnn_loss = run_experiment(rnn, "RNN")

: 

## Эксперимент 3: Сверточная нейронная сеть (CNN)

Сверточные нейронные сети специально разработаны для обработки изображений.

Они учитывают локальные пространственные зависимости и обладают свойством инвариантности к сдвигу.

In [ ]:
cnn = CNN()
cnn_acc, cnn_loss = run_experiment(cnn, "CNN")

: 

## Сравнение результатов

Результаты работы моделей представлены в таблице ниже.

In [ ]:
results = pd.DataFrame({
    "Model": ["MLP", "CNN", "RNN"],
    "Accuracy": [mlp_acc, cnn_acc, rnn_acc]
})

results

: 

## Визуализация результатов

Для анализа процесса обучения строятся графики изменения функции потерь и точности.

In [ ]:
results.plot(x="Model", y="Accuracy", kind="bar", legend=False)
plt.title("Model Comparison (Accuracy)")
plt.ylabel("Accuracy")
plt.show()

plt.plot(mlp_loss, label="MLP")
plt.plot(cnn_loss, label="CNN")
plt.plot(rnn_loss, label="RNN")

plt.title("Training Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

: 

## Вывод

В ходе работы было проведено сравнение различных архитектур нейронных сетей для задачи классификации изображений.

Полученные результаты показывают:

- Полносвязные сети демонстрируют наихудшие результаты, так как не учитывают структуру изображения.
- Рекуррентные сети показывают среднее качество, поскольку изображение не является естественной последовательностью.
- Сверточные нейронные сети достигают наилучших результатов благодаря учёту пространственных зависимостей.

Таким образом, выбор архитектуры нейронной сети должен соответствовать структуре входных данных.